In [1]:
from ta.volatility import AverageTrueRange

import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from datetime import datetime
import numpy as np

import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd

from backtesting import Strategy
from backtesting import Backtest, set_bokeh_output
set_bokeh_output(notebook=False)

from datetime import timedelta

import sys
sys.path.append('../utils')
import Util
import importlib
importlib.reload(Util)

from enum import Enum
import time

from random import randint
import random


c:\Users\kyanh\AppData\Local\Programs\Python\Python313-arm64\Lib\site-packages\backtesting\_plotting.py:55: UserWarning: Jupyter Notebook detected. Setting Bokeh output to notebook. This may not work in Jupyter clients without JavaScript support, such as old IDEs. Reset with `backtesting.set_bokeh_output(notebook=False)`.
  warnings.warn('Jupyter Notebook detected. '


Loading BokehJS ...

In [3]:
pair = 'CADCHF'

df_d1_raw = pd.read_csv(f"../Trading Data/{pair}_D1.csv",sep=',', parse_dates=['Date'], index_col='Date')
df_h4_raw = pd.read_csv(f"../Trading Data/{pair}_H4.csv",sep=',', parse_dates=['Date'], index_col='Date')
df_m15_raw = pd.read_csv(f"../Trading Data/{pair}_M15.csv",sep=',', parse_dates=['Date'], index_col='Date')

print(df_d1_raw)
print(df_h4_raw)
print(df_m15_raw)

               Open     High      Low    Close  Tickvol
Date                                                   
2010-09-23  0.95790  0.95845  0.94550  0.95260    81300
2010-09-24  0.95260  0.96180  0.94780  0.95980   103852
2010-09-27  0.96055  0.96420  0.95510  0.95655   115385
2010-09-28  0.95655  0.95765  0.94310  0.94775   126433
2010-09-29  0.94765  0.95425  0.94400  0.94620   115335
...             ...      ...      ...      ...      ...
2026-10-05  0.58075  0.58381  0.58046  0.58250    48951
2026-10-06  0.58249  0.58551  0.58186  0.58538    42990
2026-10-07  0.58540  0.58631  0.58206  0.58423    50257
2026-10-08  0.58424  0.58563  0.58294  0.58443    66525
2026-10-09  0.58444  0.58472  0.58121  0.58234    34325

[4185 rows x 5 columns]
                        Open     High      Low    Close  Tickvol
Date                                                            
2010-09-23 08:00:00  0.95790  0.95845  0.95285  0.95370    19153
2010-09-23 12:00:00  0.95370  0.95455  0.94550  0.95

In [4]:
structure_d1_window = 2
structure_m15_window = 2

date_format = '%Y-%m-%d %H:%M'
# start_date = '2023-1-1 00:00'
# end_date = '2024-3-29 00:00'
start_date = datetime.strptime('2021-1-1 00:00', date_format)
end_date = datetime.strptime('2026-9-16 00:00', date_format)
# start_date = '2022-01-1 00:00'

timezone = "Europe/London"

# nombre de jours pour les high.low
define_key_level_window = 90

gap_look_ahead = 6

# nombre de candle au dessus/ en desous de key level après breakout
breakout_candles = 9

# nombre de jours à chercher key level
key_level_lookback_candle = 100

# vérifier si 30 candle dans ke passé pour valider un valid breakout,
# un valid breakout doit avoir {breakout_candles} candles au dessus du resistant level
breakout_candles_lookback = 30

start_trade_hour = 9
stop_trade_hour = 19

atr_multiplier = 0.7
min_touched = 3

In [5]:
# process h4 data
df_h4 = df_h4_raw.loc[start_date - timedelta(days=20) : end_date]
df_h4 = Util.implement_market_structure_states_fixed_pivots(df_h4, 5)
df_h4 = df_h4.loc[start_date : end_date]

indicator_atr_14 = AverageTrueRange(df_h4['High'], df_h4['Low'], df_h4['Close'], window=14)
df_h4['ATR'] = indicator_atr_14._atr

df_h4

,Open,High,Low,Close,Tickvol,is_swing_high,is_swing_low,market_state,ATR
Date,,,,,,,,,
2021-01-04 00:00:00,0.69378,0.69573,0.69310,0.69555,12960,False,False,BULLISH,0.000000
2021-01-04 04:00:00,0.69553,0.69625,0.69513,0.69527,9826,True,False,BULLISH,0.000000
2021-01-04 08:00:00,0.69525,0.69580,0.69311,0.69377,20631,False,False,BULLISH,0.000000
2021-01-04 12:00:00,0.69377,0.69405,0.69151,0.69293,23712,False,False,BULLISH,0.000000
2021-01-04 16:00:00,0.69296,0.69395,0.68858,0.68984,32860,False,False,BULLISH,0.000000
...,...,...,...,...,...,...,...,...,...
2026-09-15 08:00:00,0.58789,0.58839,0.58749,0.58791,6852,False,False,BULLISH,0.001281
2026-09-15 12:00:00,0.58792,0.58904,0.58735,0.58878,9087,False,False,BULLISH,0.001310
2026-09-15 16:00:00,0.58879,0.58880,0.58812,0.58812,4546,False,False,BULLISH,0.001265


In [6]:
resistants_list = Util.define_resistant_level_with_atr(df_h4, define_key_level_window, atr_multiplier=atr_multiplier, min_touched=min_touched)
supports_list = Util.define_support_level_with_atr(df_h4, define_key_level_window, atr_multiplier=atr_multiplier, min_touched=min_touched)

df_h4['support_point'] = df_h4.apply(lambda row: Util.support_pos(supports_list, row), axis=1)
df_h4['is_support'] = df_h4.apply(lambda row: Util.is_support(supports_list, row), axis=1)

df_h4['resistant_point'] = df_h4.apply(lambda row: Util.resistant_pos(resistants_list, row), axis=1)
df_h4['is_resistant'] = df_h4.apply(lambda row: Util.is_resistant(resistants_list, row), axis=1)

df_h4['valid_breakout_support_levels'] = Util.is_candles_above_support_level_list(df_h4, breakout_candles, key_level_lookback_candle)
df_h4['valid_breakout_resistant_levels'] = Util.is_candles_below_resistant_level_list(df_h4, breakout_candles, key_level_lookback_candle)


In [7]:
# process m15 data
df_m15 = df_m15_raw.loc[start_date - timedelta(hours=3): end_date]
df_m15 = Util.implement_market_structure_states_fixed_pivots(df_m15, n=structure_m15_window)

m15_h4_resistant_level_list = []
m15_is_resistants_list = []
m15_is_below_resistant_list = []

m15_h4_support_level_list = []
m15_is_supports_list = []
m15_is_above_support_list = []

index_list = []

indicator_atr_m15 = AverageTrueRange(df_m15['High'], df_m15['Low'], df_m15['Close'], window=14)
df_m15['ATR'] = indicator_atr_m15._atr

df_m15 = df_m15.loc[start_date : end_date]
for i in range(len(df_h4)):
    row = df_h4.iloc[i]
    start_date_h4 = row.name
    end_date_h4 = start_date_h4 + timedelta(hours=3, minutes=59)
    df_by_day = df_m15.loc[f'{start_date_h4}' : f'{end_date_h4}']

    for index2, row2 in df_by_day.iterrows():
        m15_h4_resistant_level_list.append(df_h4.iloc[i]['resistant_point'])
        m15_is_resistants_list.append(df_h4.iloc[i]['is_resistant'])
        m15_is_below_resistant_list.append(df_h4.iloc[i]['valid_breakout_resistant_levels'])
        # affect current market states of H4 candle because the result is market state of previous D1 candle

        m15_h4_support_level_list.append(df_h4.iloc[i]['support_point'])
        m15_is_supports_list.append(df_h4.iloc[i]['is_support'])
        m15_is_above_support_list.append(df_h4.iloc[i]['valid_breakout_support_levels'])

        # ajust to daylight saving time
        dst = pd.Timestamp(index2, tz=timezone).dst()
        if dst == timedelta(0):
            index2 = index2 - timedelta(hours=1)
            index_list.append(index2)
        else:
            index_list.append(index2)

df_m15['resistant_point'] = m15_h4_resistant_level_list
df_m15['is_resistant'] = m15_is_resistants_list
df_m15['valid_breakout_resistant_levels'] = m15_is_below_resistant_list

df_m15['support_point'] = m15_h4_support_level_list
df_m15['is_support'] = m15_is_supports_list
df_m15['valid_breakout_support_levels'] = m15_is_above_support_list

df_m15.index = index_list

df_m15

,Open,High,Low,Close,Tickvol,is_swing_high,is_swing_low,market_state,ATR,resistant_point,is_resistant,valid_breakout_resistant_levels,support_point,is_support,valid_breakout_support_levels
2021-01-03 23:00:00,0.69378,0.69450,0.69310,0.69425,1856,False,True,BEARISH,0.000000,NaN,False,"[0.578, 0.585]",NaN,False,[0.587]
2021-01-03 23:15:00,0.69425,0.69472,0.69424,0.69428,451,False,False,BEARISH,0.000000,NaN,False,"[0.578, 0.585]",NaN,False,[0.587]
2021-01-03 23:30:00,0.69429,0.69438,0.69412,0.69423,595,False,False,BEARISH,0.000000,NaN,False,"[0.578, 0.585]",NaN,False,[0.587]
2021-01-03 23:45:00,0.69422,0.69422,0.69372,0.69403,767,False,True,BEARISH,0.000000,NaN,False,"[0.578, 0.585]",NaN,False,[0.587]
2021-01-04 00:00:00,0.69403,0.69450,0.69382,0.69413,914,True,False,BEARISH,0.000000,NaN,False,"[0.578, 0.585]",NaN,False,[0.587]
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-09-15 23:00:00,0.58770,0.58809,0.58770,0.58803,238,False,False,BEARISH,0.000351,NaN,False,[],NaN,False,[]
2026-09-15 23:15:00,0.58803,0.58809,0.58801,0.58808,144,False,False,BEARISH,0.000332,NaN,False,[],NaN,False,[]
2026-09-15 23:30:00,0.58805,0.58816,0.58805,0.58813,170,False,False,BEARISH,0.000316,NaN,False,[],NaN,False,[]
2026-09-15 23:45:00,0.58814,0.58820,0.58812,0.58820,147,False,False,BEARISH,0.000299,NaN,False,[],NaN,False,[]


In [8]:
def is_existed(list, value):
    for i in range(len(list)):
        if list[i] == value:
            return True
    return False

In [9]:
# using m15 market structure for entry
def apply_total_signal(df, key_level_lookback_candle = 150, gap=1, breakout_candles_lookback = 30, start_trade_hour=8, stop_trade_hour=19):
    # Initialize the 'TotalSignal' column
    df_signal_list = []
    valid_support_levels_list = []
    valid_resistant_levels_list = []
    current_day = None

    # short variables
    is_resistant_level_touched = False
    is_below_resistant_levels = False
    high_tf_condition_short_satisfied = False
    high_tf_condition_short_candle = None
    m15_market_state_bearish = False

    # long variables
    is_support_level_touched = False
    is_above_support_levels = False
    high_tf_condition_long_satisfied = False
    high_tf_condition_long_candle = None
    m15_market_state_bullish = False

    # we want market structure to switch from bearish to bullish when htf conditions are met
    # if market structure is already bullish we need to wait
    previous_market_structure = None

    resistants_levels_touched = []
    supports_levels_touched = []

    for i in range(0, len(df)):
        current_bar_date = df.index[i]
        signal = 0
        # valid resistants levels are when valid breakout and levels touched
        valid_resistant_levels = []
        valid_support_levels = []

        if current_day != current_bar_date.date() or current_bar_date.time().hour >= stop_trade_hour:
            current_day = current_bar_date.date()
            # short
            high_tf_condition_short_satisfied = False
            resistants_levels_touched = []
            is_resistant_level_touched = False
            valid_breakout_resistant_levels = []
            is_below_resistant_levels = False

            # long
            high_tf_condition_long_satisfied = False
            supports_levels_touched = []
            is_support_level_touched = False
            valid_breakout_support_levels = []
            is_above_support_levels = False

        # short
        c_m15_ms_bearish = df['market_state'].iloc[i] == 'BEARISH'

        # h4 key level conditions
        resistants_levels = Util.get_resistant_list(df, i, key_level_lookback_candle, gap, Util.TimeFrame.M15)
        for level in resistants_levels:
            if df['High'].iloc[i-1] > level:
                is_resistant_level_touched = True
                if not is_existed(resistants_levels_touched, level):
                    resistants_levels_touched.append(level)
        
        # Combine conditions
        if (is_resistant_level_touched and
            not high_tf_condition_short_satisfied):
            high_tf_condition_short_satisfied = True
            high_tf_condition_short_candle = current_bar_date
            previous_market_structure = df['market_state'].iloc[i]
            print(f"{current_bar_date} {i}")

        if (high_tf_condition_short_satisfied == True and
            c_m15_ms_bearish and
            high_tf_condition_short_candle < current_bar_date and
            previous_market_structure == 'BULLISH'):
            m15_market_state_bearish = True

        # breakout_candles_below candle needs to be below resistant level for the breakout to be valid
        valid_breakout_resistant_levels = Util.get_valid_breakout_resistant_level(df, i, breakout_candles_lookback)
        
        for valid_level in valid_breakout_resistant_levels:
            for level_touched in resistants_levels_touched:
                if valid_level == level_touched:
                    valid_resistant_levels.append(valid_level)
                    is_below_resistant_levels = True

        c_time_trade = start_trade_hour <= current_bar_date.hour and current_bar_date.hour <= stop_trade_hour
        if c_time_trade:
            if (high_tf_condition_short_satisfied == True and
                m15_market_state_bearish and
                is_below_resistant_levels):
                signal = 1

        # long
        c_m15_ms_bullish = df['market_state'].iloc[i] == 'BULLISH'
        
        # h4 support level conditions
        supports_levels = Util.get_support_list(df, i, key_level_lookback_candle, gap, Util.TimeFrame.M15)
        for level in supports_levels:
            if df['Low'].iloc[i-1] < level:
                is_support_level_touched = True
                if not is_existed(supports_levels_touched, level):
                    supports_levels_touched.append(level)

        # Combine conditions
        if (is_support_level_touched and
            not high_tf_condition_long_satisfied):
            high_tf_condition_long_satisfied = True
            m15_market_state_bullish = False
            high_tf_condition_long_candle = current_bar_date
            previous_market_structure = df['market_state'].iloc[i]
            # print(f"high_tf_condition_long_satisfied {current_bar_date} {i}")

        if (high_tf_condition_long_satisfied and
            c_m15_ms_bullish and
            high_tf_condition_long_candle < current_bar_date and
            previous_market_structure == 'BEARISH'):
            # print(f"m15_market_state_bullish {current_bar_date} {i}")
            m15_market_state_bullish = True

        # breakout_candles_above candle needs to be above support level for the breakout to be valid
        valid_breakout_support_levels = Util.get_valid_breakout_support_level(df, i, breakout_candles_lookback)
        for valid_level in valid_breakout_support_levels:
            for level_touched in supports_levels_touched:
                if valid_level == level_touched:
                    valid_support_levels.append(valid_level)
                    is_above_support_levels = True

        c_time_trade = start_trade_hour <= current_bar_date.hour and current_bar_date.hour <= stop_trade_hour
        if c_time_trade:
            if (high_tf_condition_long_satisfied == True and
                m15_market_state_bullish and
                is_above_support_levels):
                print(f"signal {current_bar_date} {i}")
                signal = 2

        if previous_market_structure != df['market_state'].iloc[i]:
            if df['market_state'].iloc[i] == 'BULLISH':
                m15_market_state_bearish = False
            if df['market_state'].iloc[i] == 'BEARISH':
                m15_market_state_bullish = False
            previous_market_structure = df['market_state'].iloc[i]    

        df_signal_list.append(signal)
        valid_resistant_levels_list.append(valid_resistant_levels)
        valid_support_levels_list.append(valid_support_levels)

    return df_signal_list, valid_resistant_levels_list, valid_support_levels_list

df_m15['TotalSignal'], df_m15['valid_resistant_levels'], df_m15['valid_support_levels'] = apply_total_signal(df=df_m15, key_level_lookback_candle=key_level_lookback_candle, gap=gap_look_ahead,
                                                                            breakout_candles_lookback=breakout_candles_lookback, start_trade_hour=start_trade_hour, stop_trade_hour=stop_trade_hour)

2021-01-03 23:00:00 0
2021-01-03 23:15:00 1
2021-01-03 23:30:00 2
2021-01-03 23:45:00 3
2021-01-04 00:00:00 4
2021-01-26 18:00:00 1600
2021-01-26 19:00:00 1604
2021-01-26 19:15:00 1605
2021-01-26 19:30:00 1606
2021-01-26 19:45:00 1607
2021-01-26 20:00:00 1608
2021-01-26 20:15:00 1609
2021-01-26 20:30:00 1610
2021-01-26 20:45:00 1611
2021-01-26 21:00:00 1612
2021-01-26 21:15:00 1613
2021-01-26 21:30:00 1614
2021-01-26 21:45:00 1615
2021-01-26 22:00:00 1616
2021-01-26 22:15:00 1617
2021-01-26 22:30:00 1618
2021-01-26 22:45:00 1619
2021-01-26 23:00:00 1620
2021-01-26 23:15:00 1621
2021-01-26 23:30:00 1622
2021-01-26 23:45:00 1623
2021-01-27 00:00:00 1624
2021-01-27 19:00:00 1700
2021-01-27 19:15:00 1701
2021-01-27 19:30:00 1702
2021-01-27 19:45:00 1703
2021-01-27 20:00:00 1704
2021-01-27 20:15:00 1705
2021-01-27 20:30:00 1706
2021-01-29 13:15:00 1869
2021-01-29 19:00:00 1892
2021-01-29 19:15:00 1893
2021-01-29 19:30:00 1894
2021-01-29 19:45:00 1895
2021-01-29 20:00:00 1896
2021-01-29 20:1

In [10]:
len(df_m15[df_m15.TotalSignal != 0])

4363

In [11]:
df_m15['pointpos'] = df_m15.apply(lambda row: Util.pointpos(row), axis=1)

In [12]:
start_date_chart = '2023-1-1 00:00'
end_date_chart = '2023-6-30 00:00'
dfpl = df_m15.loc[start_date_chart : end_date_chart]#.set_index("Gmt time")

# Create a plot with 2 rows
fig = make_subplots(rows=1, cols=1)

# Add candlestick plot on the first row
fig.add_trace(go.Candlestick(x=dfpl.index,
                             open=dfpl['Open'],
                             high=dfpl['High'],
                             low=dfpl['Low'],
                             close=dfpl['Close']),
              row=1, col=1)

# Add markers for entry point m15
fig.add_trace(go.Scatter(x=dfpl.index, y=dfpl['pointpos'], mode="markers",
                         marker=dict(size=12, color="MediumPurple"),
                         name="entry"),
              row=1, col=1)

fig.update_layout(width=1500, height=800, sliders=[])
fig.show()

In [13]:
start_date_chart = '2023-1-1 00:00'
end_date_chart = '2023-12-30 00:00'
dfpl = df_h4.loc[start_date_chart : end_date_chart]#.set_index("Gmt time")

# Create a plot with 2 rows
fig = make_subplots(rows=1, cols=1)

# Add candlestick plot on the first row
fig.add_trace(go.Candlestick(x=dfpl.index,
                             open=dfpl['Open'],
                             high=dfpl['High'],
                             low=dfpl['Low'],
                             close=dfpl['Close']),
              row=1, col=1)

#Add markers for trade entry points on the same subplot
fig.add_trace(go.Scatter(x=dfpl.index, y=dfpl['support_point'], mode="markers",
                         marker=dict(size=8, color="green"),
                         name="support point"),
              row=1, col=1)

#Add markers for trade entry points on the same subplot
fig.add_trace(go.Scatter(x=dfpl.index, y=dfpl['resistant_point'], mode="markers",
                         marker=dict(size=8, color="red"),
                         name="resistant point"),
              row=1, col=1)

fig.update_layout(width=1200, height=800, sliders=[])
fig.show()

In [14]:
class KeyLevelChochStrat(Strategy):
    mysize = 0.5
    slcoef = 2
    TPcoef = 2
    risk_percent = 0.01
    last_swing_low = None
    last_swing_high = None
    previous_swing_low = None
    previous_swing_high = None
    entry_gap_multiplier = 0.8
    last_trade_index = 0
    dfopt = None

    stoploss_info_list = []
    valid_resistant_level = None
    valid_support_level = None

    setup_duration_gap = 60
    stoploss_window = 150
    
    def init(self):
        self.current_day = None
        self.order_placed_time = None
        self.setup_duration = None
        
        self.stoploss_info_list = []
        self.valid_resistant_level = None
        self.valid_support_level = None

        super().init()

    def reset_order(self):
        # delete placed order at the end of the day if not executed
        if self.order_placed_time:
            for order in self.orders:
                order_executed= False
                for trade in self.trades:
                    if trade.tag == order.tag:
                        order_executed = True

                if order_executed == False:
                    order.cancel()

        self.order_placed_time = None
        self.setup_duration = None
        self.valid_resistant_level = None
        self.valid_support_level = None

    def get_highest_high(self):
        return self.data.High[len(self.data.High) - self.stoploss_window : len(self.data.High)].max()

    def get_lowest_low(self):
        return self.data.Low[len(self.data.Low) - self.stoploss_window  : len(self.data.Low)].min()

    def is_entry_in_valid_short_zone(self, valid_resistant_list):
        return self.data.Close[-1] < max(valid_resistant_list) + self.data['ATR'][-1] * self.entry_gap_multiplier

    def is_entry_in_valid_long_zone(self, valid_support_list):
        return self.data.Close[-1] > min(valid_support_list) - self.data['ATR'][-1] * self.entry_gap_multiplier

    def add_stoploss_info_list(self, info):
        is_added = False
        for stoploss_info in self.stoploss_info_list:
            if stoploss_info[0] == info[0] and stoploss_info[1] == info[1]:
                is_added = True
                stoploss_info[2] = info[2]

        if is_added == False:
            self.stoploss_info_list.append(info)

    def get_stoploss_info(self, key_level_type, key_level_value):
        for stoploss_info in self.stoploss_info_list:
            if stoploss_info[0] == key_level_type and stoploss_info[1] == key_level_value:
                return stoploss_info
        return None

    def dynamic_stoploss(self):
        # delete placed order at the end of the day if not executed
        if self.order_placed_time:
            for order in self.orders:
                order_executed= False
                for trade in self.trades:
                    if trade.tag == order.tag:
                        order_executed = True
                        if order.stop is not None:
                            # in long trade, short order is for sl and inverse for short trade
                            # so we identify short order for long trade to identify sl
                            if order.is_short:
                                self.add_stoploss_info_list([Util.KeyLevel.SUPPORT, self.valid_support_level, order.stop])
                            else:
                                self.add_stoploss_info_list([Util.KeyLevel.RESISTANT, self.valid_resistant_level, order.stop])


                if order_executed == False:
                    order.cancel()
                    if order.is_short:
                        sl = abs(order.stop - self.get_highest_high())
                        tp = order.stop - self.TPcoef * sl
                        position_size = int((self.risk_percent * self.equity) / sl)
                        self.sell(sl = self.get_highest_high(), tp = tp, size=position_size, stop=order.stop)
                    else:
                        sl = abs(order.stop - self.get_lowest_low())
                        tp = order.stop + self.TPcoef * sl
                        position_size = int((self.risk_percent * self.equity) / sl)
                        order = self.buy(sl = self.get_lowest_low(), tp = tp, size=position_size, stop=order.stop)

    def get_tag(self):
        return f"{self.order_placed_time}"

    def wait_3_candles_before_new_trade(self):
        return self.last_trade_index + 3
    
    def next(self):
        super().next()
        index = len(self.data.index)

        # when an order is placed, it will be valable for 12h
        if self.setup_duration is not None and index >= self.setup_duration:
            self.reset_order()

        # place sl to the lowest/highest
        self.dynamic_stoploss()

        if len(self.trades) > 0:
            self.last_trade_index = index

        if len(self.orders)==0 and index >= self.wait_3_candles_before_new_trade():
            if self.data['TotalSignal'][-1]==1:
                valid_resistant_list = Util.get_valid_resistant_level(self.dfopt, index, breakout_candles_lookback)
                if len(valid_resistant_list) > 0:
                    valid_resistant_level = max(valid_resistant_list)
                    is_entry_in_valid_zone = self.is_entry_in_valid_short_zone(valid_resistant_list)
                    planned_entry_price = 0
                    sl = 0

                    if is_entry_in_valid_zone: 
                        planned_entry_price = self.data.Close[-1]
                        sl = abs(planned_entry_price - self.get_highest_high())
                    else :
                        planned_entry_price = valid_resistant_level + self.data['ATR'][-1] * self.entry_gap_multiplier
                        sl = abs(planned_entry_price - self.get_highest_high())
    
                    sl1 = planned_entry_price + sl
                    stoploss_info = self.get_stoploss_info(Util.KeyLevel.RESISTANT, valid_resistant_level)
                    if stoploss_info is None or sl1 > stoploss_info[2]:
                        tp = self.TPcoef * sl
                        position_size = int((self.risk_percent * self.equity) / sl)
                        tp1 = planned_entry_price - tp

                        self.valid_resistant_level = valid_resistant_level
                        self.setup_duration = len(self.data.index) + self.setup_duration_gap
                        self.order_placed_time = self.data.index[-1]

                        if is_entry_in_valid_zone: 
                            self.sell(sl=sl1, tp=tp1, size=position_size, tag=self.get_tag())
                            # print(f"short {self.data.index[-1]} sl = {sl1} & sgnal = {self.signal1[-1]} & valid_resistant_level = {valid_resistant_level} & entry_price = {planned_entry_price}")
                        else:
                            self.sell(sl=sl1, tp=tp1, size=position_size, stop=planned_entry_price, tag=self.get_tag())
                            # print(f"short {self.data.index[-1]} sl = {sl1} & sgnal = {self.signal1[-1]} & valid_resistant_level = {valid_resistant_level} & planned_entry_price = {planned_entry_price}")

            if self.data['TotalSignal'][-1]==2:
                valid_support_list = Util.get_valid_support_level(self.dfopt, index, breakout_candles_lookback)
                if len(valid_support_list) > 0:
                    valid_support_level = min(valid_support_list)
                    is_entry_in_valid_zone = self.is_entry_in_valid_long_zone(valid_support_list)
                    planned_entry_price = 0
                    sl = 0
                    
                    if is_entry_in_valid_zone: 
                        planned_entry_price = self.data.Close[-1]
                        sl = abs(planned_entry_price - self.get_lowest_low())
                    else :
                        planned_entry_price = valid_support_level - self.data['ATR'][-1] * self.entry_gap_multiplier
                        sl = abs(planned_entry_price - self.get_lowest_low())

                    sl1 = planned_entry_price - sl
                    stoploss_info = self.get_stoploss_info(Util.KeyLevel.SUPPORT, valid_support_level)
                    # print(f"{stoploss_info} {valid_support_level}")
                    if stoploss_info is None or sl1 < stoploss_info[2]:
                        tp = self.TPcoef * sl
                        position_size = int((self.risk_percent * self.equity) / sl)
                        tp1 = planned_entry_price + tp

                        self.valid_support_level = valid_support_level
                        self.setup_duration = len(self.data.index) + self.setup_duration_gap
                        self.order_placed_time = self.data.index[-1]

                        if is_entry_in_valid_zone: 
                            self.buy(sl=sl1, tp=tp1, size=position_size, tag=self.get_tag())
                            # print(f"long {self.data.index[-1]} sl = {sl1} & sgnal = {self.signal1[-1]} & valid_support_level = {valid_support_level} & entry_price = {planned_entry_price}")
                        else:
                            self.buy(sl=sl1, tp=tp1, size=position_size, stop=planned_entry_price, tag=self.get_tag())
                            # print(f"long {self.data.index[-1]} sl = {sl1} & sgnal = {self.signal1[-1]} & valid_support_level = {valid_support_level} & planned_entry_price = {planned_entry_price}")
                            

In [17]:
start_date_chart = '2021-1-1 00:00'
end_date_chart = '2026-12-30 00:00'
global_dfopt = df_m15.loc[start_date_chart : end_date_chart]

bt = Backtest(global_dfopt, KeyLevelChochStrat, cash=10000, margin=0.005, finalize_trades=True) #0.0002
stats = bt.run(dfopt=global_dfopt)

print(stats)
bt.plot()


Start                     2021-01-03 23:00:00
End                       2026-09-16 00:00:00
Duration                   2081 days 01:00:00
Exposure Time [%]                    15.55392
Equity Final [$]                   8695.12261
Equity Peak [$]                    10013.6544
Return [%]                          -13.04877
Buy & Hold Return [%]                -15.2726
Return (Ann.) [%]                    -2.17143
Volatility (Ann.) [%]                 5.31732
CAGR [%]                             -2.42421
Sharpe Ratio                         -0.40837
Sortino Ratio                        -0.53798
Calmar Ratio                         -0.09497
Alpha [%]                           -12.70911
Beta                                  0.02224
Max. Drawdown [%]                   -22.86328
Avg. Drawdown [%]                   -22.86328
Max. Drawdown Duration     2055 days 06:00:00
Avg. Drawdown Duration     2055 days 06:00:00
# Trades                                   92
Win Rate [%]                      

c:\Users\kyanh\AppData\Local\Programs\Python\Python313-arm64\Lib\site-packages\backtesting\_plotting.py:141: UserWarning: Data contains too many candlesticks to plot; downsampling to '8h'. See `Backtest.plot(resample=...)`
  warnings.warn(f"Data contains too many candlesticks to plot; downsampling to {freq!r}. "


GridPlot(id='p1829', ...)

In [18]:
# add some sl tp management

class RandomStrat(Strategy):
    mysize = 0.5
    slcoef = 2
    TPcoef = 2
    risk_percent = 0.01
    trades_list = None
    stoploss_window = 150

    def is_signal(self, date):
        for i in range(0, len(self.trades_list)):
            if self.trades_list[i] == date:
                return True
        return False

    def get_highest_high(self):
        return self.data.High[len(self.data.High) - self.stoploss_window : len(self.data.High)].max()
    
    def get_lowest_low(self):
        return self.data.Low[len(self.data.Low) - self.stoploss_window : len(self.data.Low)].min()
    
    def init(self):
        super().init()

    def get_sl(self):
        return round(random.uniform(self.data['ATR'][-1]*1,self.data['ATR'][-1]*2.5), 5)

    def next(self):
        super().next()
        if self.is_signal(self.data.index[-1]) and len(self.trades) == 0 and len(self.closed_trades) < 92:
            direction = randint(1,2)
            if direction == 1:
                planned_entry_price = self.data.Close[-1]
                sl = abs(planned_entry_price - self.get_lowest_low())
                if sl == 0:
                    sl = self.get_sl()
                tp = self.TPcoef * sl

                sl1 = planned_entry_price - sl
                tp1 = planned_entry_price + tp
                position_size = int((self.risk_percent * self.equity) / sl)
                # max_affordable_size = int((self.equity * 0.99) / planned_entry_price)
                # print(f'{position_size} {max_affordable_size}')

                self.buy(sl=sl1, tp=tp1, size=position_size)
            else:
                planned_entry_price = self.data.Close[-1]
                sl = abs(planned_entry_price - self.get_highest_high())
                if sl == 0:
                    sl = self.get_sl()
                tp = self.TPcoef * sl

                sl1 = planned_entry_price + sl
                tp1 = planned_entry_price - tp
                position_size = int((self.risk_percent * self.equity) / sl)
                # max_affordable_size = int((self.equity * 0.99) / planned_entry_price)
                # print(f'{position_size} {max_affordable_size}')

                self.sell(sl=sl1, tp=tp1, size=position_size)

In [24]:
trades_list = []
while len(trades_list) != 125:
    random_index = randint(0, len(df_m15))
    candle = df_m15.iloc[random_index]
    # exclu Saturday and Sunday
    if candle.name.weekday() < 5:
        if 9 <= candle.name.hour and candle.name.hour <= 19:
            trades_list.append(candle.name)
trades_list.sort()

bt = Backtest(df_m15, RandomStrat, cash=10000, margin=0.005, finalize_trades=True) #0.0002
stats = bt.run(trades_list=trades_list)

print(stats)

bt.plot()

c:\Users\kyanh\AppData\Local\Programs\Python\Python313-arm64\Lib\site-packages\backtesting\backtesting.py:1009: UserWarning: time=113983: Broker canceled the order due to insufficient margin (equity=9335.49, margin_available=9335.49).
  warnings.warn(


Start                     2021-01-03 23:00:00
End                       2026-09-16 00:00:00
Duration                   2081 days 01:00:00
Exposure Time [%]                    23.00327
Equity Final [$]                   9688.82734
Equity Peak [$]                   10318.47783
Return [%]                           -3.11173
Buy & Hold Return [%]                -15.2726
Return (Ann.) [%]                     -0.4951
Volatility (Ann.) [%]                 5.73493
CAGR [%]                             -0.55329
Sharpe Ratio                         -0.08633
Sortino Ratio                        -0.12881
Calmar Ratio                         -0.03814
Alpha [%]                            -3.61108
Beta                                  -0.0327
Max. Drawdown [%]                   -12.98034
Avg. Drawdown [%]                    -1.31898
Max. Drawdown Duration     1013 days 13:00:00
Avg. Drawdown Duration       75 days 21:28:00
# Trades                                   92
Win Rate [%]                      

c:\Users\kyanh\AppData\Local\Programs\Python\Python313-arm64\Lib\site-packages\backtesting\_plotting.py:141: UserWarning: Data contains too many candlesticks to plot; downsampling to '8h'. See `Backtest.plot(resample=...)`
  warnings.warn(f"Data contains too many candlesticks to plot; downsampling to {freq!r}. "


GridPlot(id='p3533', ...)

In [25]:
i = 0
expectancy_list = []
winrate_list = []
number_trades = []

while len(expectancy_list) < 1000:
    trades_list = []
    while len(trades_list) != 125:
        random_index = randint(0, len(df_m15)-1)
        candle = df_m15.iloc[random_index -1]
        # exclu Saturday and Sunday
        if candle.name.weekday() < 5:
            if 9 <= candle.name.hour and candle.name.hour <= 19:
                trades_list.append(candle.name)
    trades_list.sort()

    bt = Backtest(df_m15, RandomStrat, cash=10000, margin=0.0009, finalize_trades=True) #0.0002
    stats = bt.run(trades_list=trades_list)
    if stats['# Trades'] == 92:
        expectancy_list.append(stats['Expectancy [%]'])
        winrate_list.append(stats['Win Rate [%]'])
        number_trades.append(stats['# Trades'])
        print(f'{i} {len(expectancy_list)} {stats['Expectancy [%]']} Done')
    i += 1

print(expectancy_list)

0 1 -0.00881334090899309 Done
1 2 -0.11712201901315354 Done
3 3 -0.12399877640231036 Done
7 4 -0.04261260222988561 Done
10 5 -0.14737783748654315 Done
11 6 -0.041967979694688656 Done
12 7 0.0013183713930174585 Done
15 8 -0.03552873677597092 Done
16 9 -0.0223855980882275 Done
17 10 0.08997040131637164 Done
19 11 -0.16236075345679168 Done
20 12 -0.0214471486341899 Done
21 13 0.0471007810727827 Done
22 14 -0.0796222904500601 Done
26 15 0.05238179137494241 Done
31 16 -0.05688316436611959 Done
32 17 -0.03278968068854319 Done
33 18 -0.13326357278840065 Done
34 19 -0.05219414041847477 Done
35 20 -0.17113116009130022 Done
36 21 -0.13044348689360938 Done
37 22 -0.06761674724225979 Done
38 23 -0.03947140349411491 Done
39 24 -0.1302455097809352 Done
44 25 -0.09280146318316039 Done
45 26 0.0014824108019892786 Done
46 27 0.04256199380299511 Done
47 28 0.0007727761368052789 Done
50 29 -0.09244627828403786 Done
51 30 -0.08400287891881374 Done
52 31 0.18716682567957577 Done
55 32 0.03652986485078688 D

In [26]:
print(len(expectancy_list))

p95 = np.percentile(expectancy_list, 95)                  # ngưỡng phân vị 95
print(f"p95 = {p95}")
# max_expectancy = max(expectancy_list_2)
print(f"max = {max(expectancy_list)}")
mean_expectancy = np.mean(expectancy_list)
print(f"mean = {mean_expectancy}")
print(f"min = {min(expectancy_list)}")
print(f"gap = {-0.02364 - mean_expectancy}")

df_result = pd.DataFrame({'expectancy' : expectancy_list})
rank = (df_result['expectancy'] <= -0.02364).mean() * 100 
print(f"rank = {rank}")
p_value = ((df_result['expectancy'] >= -0.02364).sum() + 1) / (len(df_result['expectancy']) + 1)
print(f"p_value = {p_value}")

print(len(winrate_list))

p95_winrate = np.percentile(winrate_list, 95)                  # ngưỡng phân vị 95
print(f"p95 = {p95_winrate}")
# max_expectancy = max(expectancy_list_2)
print(f"max = {max(winrate_list)}")
mean_winrate = np.mean(winrate_list)
print(f"mean = {mean_winrate}")
print(f"min = {min(winrate_list)}")
print(f"gap = {28.26087 - mean_winrate}")

1000
p95 = 0.08755884672572535
max = 0.21408115114562226
mean = -0.042938653049123275
min = -0.277442104297038
gap = 0.019298653049123274
rank = 60.699999999999996
p_value = 0.3936063936063936
1000
p95 = 39.130434782608695
max = 48.91304347826087
mean = 31.058695652173917
min = 17.391304347826086
gap = -2.797825652173916
